In [0]:
dbutils.widgets.text("catalog", "scl")
catalog = dbutils.widgets.get("catalog")

#q01

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.q01 AS
SELECT
    YEAR(order_date) AS order_year,
    SUM(extended_price * (1 - discount)) AS net_revenue,
    COUNT(DISTINCT order_key) AS total_orders,
    COUNT(DISTINCT customer_key) AS active_customers,
    ROUND(SUM(extended_price * (1 - discount)) / COUNT(DISTINCT order_key), 2) AS avg_order_value
FROM {catalog}.gold.fact_sales
GROUP BY YEAR(order_date)
ORDER BY order_year
""")

display(spark.table(f"{catalog}.gold.q01"))

#q02

In [0]:
#monthly revenue trend across all year
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q02 AS
    SELECT
        YEAR(order_date) AS order_year,
        DATE_FORMAT(order_date, 'MMMM') AS order_month,
        DATE_TRUNC('month', order_date) AS month_start,
        SUM(extended_price * (1 - discount)) AS net_revenue,
        COUNT(DISTINCT order_key) AS total_orders,
        COUNT(DISTINCT customer_key) AS active_customers,
        ROUND(SUM(extended_price * (1 - discount)) / COUNT(DISTINCT order_key), 2) AS avg_order_value
    FROM {catalog}.gold.fact_sales
    GROUP BY YEAR(order_date), DATE_FORMAT(order_date, 'MMMM'), DATE_TRUNC('month', order_date)
    ORDER BY order_year, month_start
""")

display(spark.table(f"{catalog}.gold.q02"))

#q03

In [0]:
#annual revenue with yoy change
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q03 AS
    WITH annual AS (
        SELECT
            YEAR(order_date) AS order_year,
            SUM(extended_price * (1 - discount)) AS net_revenue,
            COUNT(DISTINCT order_key) AS total_orders,
            COUNT(DISTINCT customer_key) AS active_customers
        FROM {catalog}.gold.fact_sales
        GROUP BY YEAR(order_date)
    )
    SELECT
        order_year,
        net_revenue,
        total_orders,
        active_customers,
        LAG(net_revenue) OVER (ORDER BY order_year) AS prev_year_revenue,
        ROUND(net_revenue - LAG(net_revenue) OVER (ORDER BY order_year), 2) AS yoy_change,
        ROUND(
            (net_revenue - LAG(net_revenue) OVER (ORDER BY order_year)) /
            LAG(net_revenue) OVER (ORDER BY order_year) * 100, 2
        ) AS yoy_pct_change
    FROM annual
    ORDER BY order_year
""")

display(spark.table(f"{catalog}.gold.q03"))

#q04

In [0]:
#revenue by customer market segment
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q04 AS
    SELECT
        c.market_segment,
        SUM(f.extended_price * (1 - f.discount)) AS net_revenue,
        COUNT(DISTINCT f.order_key) AS total_orders,
        COUNT(DISTINCT f.customer_key) AS active_customers,
        ROUND(SUM(f.extended_price * (1 - f.discount)) / COUNT(DISTINCT f.order_key), 2) AS avg_order_value
    FROM {catalog}.gold.fact_sales f
    LEFT JOIN {catalog}.gold.dim_customer c
        ON f.customer_key = c.customer_key
    GROUP BY c.market_segment
    ORDER BY net_revenue DESC
""")

display(spark.table(f"{catalog}.gold.q04"))

#q05

In [0]:
#top 10 customers by net revenue
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q05 AS
    SELECT
        c.customer_key,
        c.customer_name,
        c.market_segment,
        SUM(f.extended_price * (1 - f.discount)) AS net_revenue,
        COUNT(DISTINCT f.order_key) AS total_orders,
        ROUND(SUM(f.extended_price * (1 - f.discount)) / COUNT(DISTINCT f.order_key), 2) AS avg_order_value
    FROM {catalog}.gold.fact_sales f
    JOIN {catalog}.gold.dim_customer c
        ON f.customer_key = c.customer_key
    GROUP BY c.customer_key, c.customer_name, c.market_segment
    ORDER BY net_revenue DESC
    LIMIT 10
""")

display(spark.table(f"{catalog}.gold.q05"))

#q06

In [0]:
#top 10 customers as a share of total revenue 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q06 AS
    WITH customer_revenue AS (
        SELECT
            c.customer_key,
            c.customer_name,
            c.market_segment,
            SUM(f.extended_price * (1 - f.discount)) AS net_revenue
        FROM {catalog}.gold.fact_sales f
        JOIN {catalog}.gold.dim_customer c
            ON f.customer_key = c.customer_key
        GROUP BY c.customer_key, c.customer_name, c.market_segment
    ),
    totals AS (
        SELECT SUM(net_revenue) AS total_revenue FROM customer_revenue
    )
    SELECT
        cr.customer_key,
        cr.customer_name,
        cr.market_segment,
        cr.net_revenue,
        ROUND(cr.net_revenue / t.total_revenue * 100, 4) AS revenue_share_pct,
        t.total_revenue
    FROM customer_revenue cr
    CROSS JOIN totals t
    ORDER BY cr.net_revenue DESC
    LIMIT 10
""")

display(spark.table(f"{catalog}.gold.q06"))

#q07

In [0]:
#top 10 products by net revenue 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q07 AS
    SELECT
        p.part_key,
        p.part_name,
        p.brand,
        p.manufacturer,
        SUM(f.extended_price * (1 - f.discount)) AS net_revenue,
        SUM(f.quantity) AS total_quantity_sold,
        COUNT(DISTINCT f.order_key) AS total_orders
    FROM {catalog}.gold.fact_sales f
    JOIN {catalog}.gold.dim_product p
        ON f.part_key = p.part_key
    GROUP BY p.part_key, p.part_name, p.brand, p.manufacturer
    ORDER BY net_revenue DESC
    LIMIT 10
""")

display(spark.table(f"{catalog}.gold.q07"))

#q08

In [0]:
#bottom 10 products by net revenue 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q08 AS
    SELECT
        p.part_key,
        p.part_name,
        p.brand,
        p.manufacturer,
        SUM(f.extended_price * (1 - f.discount)) AS net_revenue,
        SUM(f.quantity) AS total_quantity_sold,
        COUNT(DISTINCT f.order_key) AS total_orders
    FROM {catalog}.gold.fact_sales f
    JOIN {catalog}.gold.dim_product p
        ON f.part_key = p.part_key
    GROUP BY p.part_key, p.part_name, p.brand, p.manufacturer
    ORDER BY net_revenue ASC
    LIMIT 10
""")

display(spark.table(f"{catalog}.gold.q08"))

#q09

In [0]:
#revenue by product type 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q09 AS
    SELECT
        p.part_type,
        SUM(f.extended_price * (1 - f.discount)) AS net_revenue,
        SUM(f.quantity) AS total_quantity_sold,
        COUNT(DISTINCT f.order_key) AS total_orders,
        COUNT(DISTINCT f.part_key) AS unique_products,
        ROUND(SUM(f.extended_price * (1 - f.discount)) / COUNT(DISTINCT f.order_key), 2) AS avg_order_value
    FROM {catalog}.gold.fact_sales f
    JOIN {catalog}.gold.dim_product p
        ON f.part_key = p.part_key
    GROUP BY p.part_type
    ORDER BY net_revenue DESC
""")

display(spark.table(f"{catalog}.gold.q09"))

#q10

In [0]:
#top 10 suppliers by net revenue 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q10 AS
    SELECT
        s.supplier_key,
        s.supplier_name,
        SUM(f.extended_price * (1 - f.discount)) AS net_revenue,
        SUM(f.quantity) AS total_quantity_supplied,
        COUNT(DISTINCT f.order_key) AS total_orders,
        COUNT(DISTINCT f.part_key) AS unique_products_supplied
    FROM {catalog}.gold.fact_sales f
    JOIN {catalog}.gold.dim_supplier s
        ON f.supplier_key = s.supplier_key
    GROUP BY s.supplier_key, s.supplier_name
    ORDER BY net_revenue DESC
    LIMIT 10
""")

display(spark.table(f"{catalog}.gold.q10"))

#q11

In [0]:
#revenue share of top 10 suppliers 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q11 AS
    WITH supplier_revenue AS (
        SELECT
            s.supplier_key,
            s.supplier_name,
            SUM(f.extended_price * (1 - f.discount)) AS net_revenue
        FROM {catalog}.gold.fact_sales f
        JOIN {catalog}.gold.dim_supplier s
            ON f.supplier_key = s.supplier_key
        GROUP BY s.supplier_key, s.supplier_name
    ),
    totals AS (
        SELECT SUM(net_revenue) AS total_revenue FROM supplier_revenue
    )
    SELECT
        sr.supplier_key,
        sr.supplier_name,
        sr.net_revenue,
        ROUND(sr.net_revenue / t.total_revenue * 100, 4) AS revenue_share_pct,
        t.total_revenue
    FROM supplier_revenue sr
    CROSS JOIN totals t
    ORDER BY sr.net_revenue DESC
    LIMIT 10
""")

display(spark.table(f"{catalog}.gold.q11"))

#q12

In [0]:
#revenue by customer region 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q12 AS
    SELECT
        g.region_name,
        g.nation_name,
        SUM(f.extended_price * (1 - f.discount)) AS net_revenue,
        COUNT(DISTINCT f.order_key) AS total_orders,
        COUNT(DISTINCT f.customer_key) AS active_customers,
        ROUND(SUM(f.extended_price * (1 - f.discount)) / COUNT(DISTINCT f.order_key), 2) AS avg_order_value
    FROM {catalog}.gold.fact_sales f
    JOIN {catalog}.gold.dim_customer c
        ON f.customer_key = c.customer_key
    JOIN {catalog}.gold.dim_geography g
        ON c.nation_key = g.nation_key
    GROUP BY g.region_name, g.nation_name
    ORDER BY net_revenue DESC
""")

display(spark.table(f"{catalog}.gold.q12"))

#q13

In [0]:
#revenue by customer nation 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q13 AS
    SELECT
        g.nation_name,
        g.region_name,
        SUM(f.extended_price * (1 - f.discount)) AS net_revenue,
        COUNT(DISTINCT f.order_key) AS total_orders,
        COUNT(DISTINCT f.customer_key) AS active_customers,
        ROUND(SUM(f.extended_price * (1 - f.discount)) / COUNT(DISTINCT f.order_key), 2) AS avg_order_value
    FROM {catalog}.gold.fact_sales f
    JOIN {catalog}.gold.dim_customer c
        ON f.customer_key = c.customer_key
    JOIN {catalog}.gold.dim_geography g
        ON c.nation_key = g.nation_key
    GROUP BY g.nation_name, g.region_name
    ORDER BY net_revenue DESC
""")

display(spark.table(f"{catalog}.gold.q13"))

   
#q14

In [0]:
#revenue and volume split by return flag 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q14 AS
    SELECT
        return_flag,
        COUNT(*) AS total_line_items,
        SUM(quantity) AS total_volume,
        SUM(extended_price) AS gross_revenue,
        SUM(extended_price * (1 - discount)) AS net_revenue,
        ROUND(SUM(extended_price * (1 - discount)) / SUM(SUM(extended_price * (1 - discount))) OVER () * 100, 2) AS net_revenue_share_pct,
        ROUND(SUM(quantity) / SUM(SUM(quantity)) OVER () * 100, 2) AS volume_share_pct,
        COUNT(DISTINCT order_key) AS total_orders
    FROM {catalog}.gold.fact_sales
    GROUP BY return_flag
    ORDER BY net_revenue DESC
""")

display(spark.table(f"{catalog}.gold.q14"))

   
#q15

In [0]:
#average discount by market segment 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q15 AS
    SELECT
        c.market_segment,
        COUNT(*) AS total_line_items,
        ROUND(AVG(f.discount) * 100, 2) AS avg_discount_pct,
        ROUND(MIN(f.discount) * 100, 2) AS min_discount_pct,
        ROUND(MAX(f.discount) * 100, 2) AS max_discount_pct,
        SUM(f.extended_price * f.discount) AS total_discount_amount,
        SUM(f.extended_price * (1 - f.discount)) AS net_revenue
    FROM {catalog}.gold.fact_sales f
    JOIN {catalog}.gold.dim_customer c
        ON f.customer_key = c.customer_key
    GROUP BY c.market_segment
    ORDER BY avg_discount_pct DESC
""")

display(spark.table(f"{catalog}.gold.q15"))

   
#q16

In [0]:
#average ship to receipt days by shipping mode 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q16 AS
    SELECT
        ship_mode,
        COUNT(*) AS total_shipments,
        ROUND(AVG(DATEDIFF(receipt_date, ship_date)), 2) AS avg_ship_to_receipt_days,
        ROUND(MIN(DATEDIFF(receipt_date, ship_date)), 0) AS min_days,
        ROUND(MAX(DATEDIFF(receipt_date, ship_date)), 0) AS max_days,
        SUM(extended_price * (1 - discount)) AS net_revenue
    FROM {catalog}.gold.fact_sales
    GROUP BY ship_mode
    ORDER BY avg_ship_to_receipt_days DESC
""")

display(spark.table(f"{catalog}.gold.q16"))

   
#q17

In [0]:
#order count and revenue by order priority 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q17 AS
    SELECT
        order_priority,
        COUNT(DISTINCT order_key) AS total_orders,
        COUNT(*) AS total_line_items,
        SUM(extended_price * (1 - discount)) AS net_revenue,
        ROUND(SUM(extended_price * (1 - discount)) / COUNT(DISTINCT order_key), 2) AS avg_order_value,
        COUNT(DISTINCT customer_key) AS active_customers
    FROM {catalog}.gold.fact_sales
    GROUP BY order_priority
    ORDER BY net_revenue DESC
""")

display(spark.table(f"{catalog}.gold.q17"))

#q18

In [0]:
#monthly breakdown of order status
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q18 AS
    SELECT
        YEAR(order_date) AS order_year,
        MONTH(order_date) AS order_month,
        DATE_FORMAT(order_date, 'MMMM') AS month_name,
        DATE_TRUNC('month', order_date) AS month_start,
        order_status,
        COUNT(DISTINCT order_key) AS total_orders,
        SUM(extended_price * (1 - discount)) AS net_revenue,
        SUM(quantity) AS total_volume
    FROM {catalog}.gold.fact_sales
    GROUP BY YEAR(order_date), MONTH(order_date), DATE_FORMAT(order_date, 'MMMM'), DATE_TRUNC('month', order_date), order_status
    ORDER BY order_year, order_month, order_status
""")

display(spark.table(f"{catalog}.gold.q18"))

#q19

In [0]:
#product revenue ranking within each market segment 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q19 AS
    WITH product_segment_revenue AS (
        SELECT
            c.market_segment,
            p.part_key,
            p.part_name,
            p.brand,
            SUM(f.extended_price * (1 - f.discount)) AS net_revenue,
            SUM(f.quantity) AS total_quantity_sold,
            COUNT(DISTINCT f.order_key) AS total_orders
        FROM {catalog}.gold.fact_sales f
        JOIN {catalog}.gold.dim_customer c
            ON f.customer_key = c.customer_key
        JOIN {catalog}.gold.dim_product p
            ON f.part_key = p.part_key
        GROUP BY c.market_segment, p.part_key, p.part_name, p.brand
    )
    SELECT
        market_segment,
        part_key,
        part_name,
        brand,
        net_revenue,
        total_quantity_sold,
        total_orders,
        RANK() OVER (PARTITION BY market_segment ORDER BY net_revenue DESC) AS revenue_rank
    FROM product_segment_revenue
    ORDER BY market_segment, revenue_rank
""")

display(spark.table(f"{catalog}.gold.q19"))

   
#q20

In [0]:
#revenue comparison: customer geography vs supplier geography by region 
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.q20 AS
    WITH customer_region AS (
        SELECT
            g.region_name AS customer_region,
            SUM(f.extended_price * (1 - f.discount)) AS customer_revenue,
            COUNT(DISTINCT f.order_key) AS customer_orders,
            COUNT(DISTINCT f.customer_key) AS customer_count
        FROM {catalog}.gold.fact_sales f
        JOIN {catalog}.gold.dim_customer c
            ON f.customer_key = c.customer_key
        JOIN {catalog}.gold.dim_geography g
            ON c.nation_key = g.nation_key
        GROUP BY g.region_name
    ),
    supplier_region AS (
        SELECT
            g.region_name AS supplier_region,
            SUM(f.extended_price * (1 - f.discount)) AS supplier_revenue,
            COUNT(DISTINCT f.order_key) AS supplier_orders,
            COUNT(DISTINCT f.supplier_key) AS supplier_count
        FROM {catalog}.gold.fact_sales f
        JOIN {catalog}.gold.dim_supplier s
            ON f.supplier_key = s.supplier_key
        JOIN {catalog}.gold.dim_geography g
            ON s.nation_key = g.nation_key
        GROUP BY g.region_name
    )
    SELECT
        COALESCE(cr.customer_region, sr.supplier_region) AS region_name,
        cr.customer_revenue,
        cr.customer_orders,
        cr.customer_count,
        sr.supplier_revenue,
        sr.supplier_orders,
        sr.supplier_count,
        ROUND(cr.customer_revenue - sr.supplier_revenue, 2) AS revenue_difference,
        ROUND((cr.customer_revenue - sr.supplier_revenue) / sr.supplier_revenue * 100, 2) AS pct_difference
    FROM customer_region cr
    FULL OUTER JOIN supplier_region sr
        ON cr.customer_region = sr.supplier_region
    ORDER BY region_name
""")

display(spark.table(f"{catalog}.gold.q20"))